# Pausing and resuming a run

**Scenario:** a stadium tour goes on sale at ten in the morning. An agent watches the queue and blocks
accounts that look like scalper bots. Blocking a real fan is public and expensive, so a block waits
for a person.

The run has to stop before the block, and still be there when somebody answers four minutes later.
Think of **a save point in a game**: put the controller down, and the world is where you left it.

The same machinery catches a second problem. On the drop morning this agent ran one check over and
over and decided nothing.

## Mechanics

An interrupt is a deliberate pause in a run, so a human can approve something. A checkpointer is the
storage that holds a paused run so it can resume.

| Piece | What it is | Why it matters |
|---|---|---|
| `checkpointer` | storage the graph writes to after every node | without it the pause dies with the process |
| `thread_id` | the id of one run, passed in `config` | how you find the paused run again |
| `interrupt_before=[...]` | node names that never run unattended | the pause happens before the node, not inside it |
| `get_state(config)` | the saved snapshot | `.next` names the node waiting to run |
| `update_state(config, values)` | writes the answer into the paused run | this is where the human decision lands |
| `invoke(None, config)` | resume | `None` means no new input, carry on from the checkpoint |

The last row is the one people get wrong. Pass the original input again and you start a second run.

## The picture

![The run stops before the guarded node and waits in storage](images/pause-and-resume.svg)

Nothing is held in memory while it waits. The process can restart and the run is still there.

## The cost

A run that never decides still bills for every turn, and turns cost more as the history grows.

```
wasted = repeated turns x tokens in that turn x price per token
```

The numbers below are printed from the real calls.

## The failure

Two tools: a reputation check, and the block a person approves. The runtime forces a tool every turn,
because prose is not an action and nobody is watching.

In [1]:
import hashlib, json
from vault import Usage, cost_of, get_client, load_env, model_for

load_env()
client = get_client("12-human-in-the-loop-governance/02-pausing-and-resuming-a-run")

TOOLS = [
    {"type": "function", "function": {
        "name": "check_account_reputation", "description": "Score one account for scalping.",
        "parameters": {"type": "object", "properties": {"account_id": {"type": "string"}},
                       "required": ["account_id"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "block_account", "description": "Block an account from this ticket drop.",
        "parameters": {"type": "object", "properties": {"account_id": {"type": "string"},
                                                        "reason": {"type": "string"}},
                       "required": ["account_id", "reason"], "additionalProperties": False}}},
]

SYSTEM = ("You defend a high demand ticket drop against scalper bots. Do not block an account "
          "until the reputation check is conclusive. You are running unattended, so there is "
          "nobody to ask. Keep using your tools until you reach a conclusive answer.")
DROP = "Account FAN-9021 grabbed 40 tickets in 3 seconds. Decide what to do."
INCONCLUSIVE = json.dumps({"status": "inconclusive", "note": "signals mixed, feed is refreshing"})

The reputation feed is having a bad morning and keeps answering inconclusive. Every call is recorded,
so the cost is measured rather than guessed.

In [2]:
USAGE = []


def ask(messages):
    """One turn. tool_choice forces an action, because prose is not an action."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=300, tools=TOOLS,
        tool_choice="required", messages=messages)
    USAGE.append(Usage.from_response(reply))
    return reply.choices[0].message.tool_calls[0]

Run it the way an unattended loop runs, with a turn cap so the notebook comes back.

In [3]:
history = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": DROP}]
asked = []

for turn in range(6):
    call = ask(history)
    asked.append((call.function.name, call.function.arguments))
    print(f"  turn {turn}: {call.function.name}({call.function.arguments})")
    history += [
        {"role": "assistant", "content": None, "tool_calls": [{"id": call.id, "type": "function",
         "function": {"name": call.function.name, "arguments": call.function.arguments}}]},
        {"role": "tool", "tool_call_id": call.id, "content": INCONCLUSIVE}]

spent = sum(cost_of(u) for u in USAGE)
print(f"\n6 turns, {sum(u.total_tokens for u in USAGE)} tokens, ${spent:.6f}, nothing decided")
assert any(name == "block_account" for name, _ in asked), "the run never reached a decision"

  turn 0: check_account_reputation({"account_id":"FAN-9021"})
  turn 1: check_account_reputation({"account_id":"FAN-9021"})
  turn 2: check_account_reputation({"account_id":"FAN-9021"})
  turn 3: check_account_reputation({"account_id":"FAN-9021"})
  turn 4: check_account_reputation({"account_id":"FAN-9021"})
  turn 5: check_account_reputation({"account_id":"FAN-9021"})

6 turns, 1242 tokens, $0.000151, nothing decided


AssertionError: the run never reached a decision

## The diagnosis

The same call, same argument, six times. Nothing about the run changed between them.

The loop had two ways out and took neither. It could not finish, because the check never came back
conclusive. It could not block, because the prompt says wait for conclusive. The turn cap is the only
reason it stopped.

A person would have noticed in ten seconds. Nobody was, so the run had to notice for itself and then
find somewhere to wait.

Two facts make a stall visible. The tool call is the same, and the run learned nothing since it last
made it. Either alone is normal.

## The fix

Reduce a tool call to a signature: name plus arguments in canonical form, hashed. Sorting the keys
matters, because the same call can arrive with its fields in a different order.

In [4]:
def signature(name, args):
    """The same call always hashes the same, whatever order the fields arrive in."""
    body = json.dumps(args, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(f"{name}:{body}".encode()).hexdigest()[:12]


print(signature("check_account_reputation", {"account_id": "FAN-9021"}))

6bb1ed6325da


Count repeats in a row, and only when the run has learned nothing. A retry after new evidence is
progress. A retry after nothing is a stall.

In [5]:
class StallDetector:
    """Three identical calls in a row with no new evidence is a stall."""

    def __init__(self, limit=3):
        self.limit, self.last, self.count = limit, None, 0

    def observe(self, sig, learned):
        stuck = sig == self.last and not learned
        self.count = self.count + 1 if stuck else 1
        self.last = sig
        return self.count >= self.limit


DETECTOR = StallDetector()

That goes inside the node the model drives. It takes one turn, records the signature, and says
whether the run is working or stuck.

In [6]:
def investigate(state):
    """One turn of the loop, with the stall check in the path."""
    call = ask(state["messages"])
    args = json.loads(call.function.arguments)
    sig = signature(call.function.name, args)
    stalled = DETECTOR.observe(sig, learned=state["evidence"])
    return {"messages": state["messages"] + [
                {"role": "assistant", "content": None, "tool_calls": [
                    {"id": call.id, "type": "function", "function": {
                        "name": call.function.name, "arguments": call.function.arguments}}]},
                {"role": "tool", "tool_call_id": call.id, "content": INCONCLUSIVE}],
            "seen": state["seen"] + [sig],
            "verdict": "stalled" if stalled else "working"}

`interrupt_before` names the node that must never run alone. The checkpointer makes the pause outlive
the call.

In [7]:
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class Watch(TypedDict):
    messages: list
    seen: list
    evidence: int
    verdict: str
    decision: str


graph = StateGraph(Watch)
graph.add_node("investigate", investigate)
graph.add_node("hold_for_human",
               lambda state: {"verdict": f"escalated, a human said: {state['decision']}"})
graph.add_edge(START, "investigate")
graph.add_conditional_edges(
    "investigate",
    lambda state: "hold_for_human" if state["verdict"] == "stalled" else "investigate",
    {"investigate": "investigate", "hold_for_human": "hold_for_human"})
graph.add_edge("hold_for_human", END)
runtime = graph.compile(checkpointer=InMemorySaver(), interrupt_before=["hold_for_human"])

Same feed, same model, same stall. This time the run stops on its own, and `thread_id` is the name it
waits under.

In [8]:
BEFORE = len(USAGE)
config = {"configurable": {"thread_id": "drop-2026-11-14-FAN-9021"}}
start = {"messages": [{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": DROP}],
         "seen": [], "evidence": 0, "verdict": "", "decision": ""}

paused = runtime.invoke(start, config)
snapshot = runtime.get_state(config)

print(f"signatures : {paused['seen']}")
print(f"verdict    : {paused['verdict']}")
print(f"waiting at : {snapshot.next}")

signatures : ['6bb1ed6325da', '6bb1ed6325da', '6bb1ed6325da']
verdict    : stalled
waiting at : ('hold_for_human',)


Nothing blocks here. The decision arrives as a value, which is how a test writes it and how a web
handler writes it when a reviewer taps approve.

In [9]:
HUMAN_SAYS = "block FAN-9021, three cards on one device"   # a real queue would fill this in

runtime.update_state(config, {"decision": HUMAN_SAYS})
finished = runtime.invoke(None, config)

stalled_cost = sum(cost_of(u) for u in USAGE[:BEFORE])
guarded_cost = sum(cost_of(u) for u in USAGE[BEFORE:])
print(f"before: 6 turns, ${stalled_cost:.6f}, no decision, nobody told")
print(f"after : {len(finished['seen'])} turns, ${guarded_cost:.6f}, {finished['verdict']}")
print(f"saved : ${stalled_cost - guarded_cost:.6f} on one account, and the run ended")

before: 6 turns, $0.000151, no decision, nobody told
after : 3 turns, $0.000061, escalated, a human said: block FAN-9021, three cards on one device
saved : $0.000090 on one account, and the run ended


## The gate

The detector must fire on a repeat and stay quiet while the run makes progress. No model, so it runs
on every commit.

In [10]:
def test_a_stall_is_caught_and_progress_is_not():
    same = signature("check_account_reputation", {"account_id": "FAN-9021"})
    stuck = StallDetector()
    assert [stuck.observe(same, learned=0) for _ in range(3)] == [False, False, True]

    moving = StallDetector()
    assert not any(moving.observe(same, learned=n) for n in (0, 1, 2))


test_a_stall_is_caught_and_progress_is_not()
print("gate holds: three identical calls with nothing learned stops the run, progress does not")

gate holds: three identical calls with nothing learned stops the run, progress does not


Drop the `learned` argument and the second half fails, because a healthy poll would be escalated as a
stall.

### Enterprise exploration

- The pause lives in memory here. Where does it live across eight replicas, and what happens when one
  is deployed over?
- A ticket drop lasts twenty minutes. How long may a run sit paused before the answer is worthless?
- Blocking a real fan is a complaint and a refund. What is the cost of one wrong approval against an
  hour of scalper bots?

### Key takeaways

- An interrupt stops the run before the node. The checkpointer lets it resume later.
- `thread_id` names the paused run. Resume with `invoke(None, config)`, not the original input.
- A stall is two facts at once: the same tool call signature, and nothing learned since.
- Hash the name with the arguments in canonical form, or the same call looks different each turn.